<a href="https://colab.research.google.com/github/Viswas-Vinayakumar/AI-search/blob/main/encoder_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inside the semantic encoder, step by step

This is the **exact model the coach-search app uses**: `paraphrase-multilingual-MiniLM-L12-v2`.
In the app, all of this happens in **one line**: `model.encode(text, normalize_embeddings=True)`.
Here we open that line and look at every step.

| Step | What happens |
|---|---|
| 1 | The query (one short sentence) |
| 2 | **Tokenization**: text → pieces → id numbers |
| 3 | **Embedding lookup**: each id → its starting vector (one row of a big table) |
| 4 | **12 transformer layers**: every token looks at every other token and updates its vector |
| 5 | **Mean pooling**: many token vectors → one sentence vector |
| 6 | **Normalization**: make the vector length 1 |
| 7 | **Check**: our hand-made vector = what the app gets |
| 8 | **Compare**: query vs a few coach sentences |

**Run it in Google Colab:** Runtime → Run all. The first run downloads the model (about 470 MB, 1 minute). No GPU needed.

The example sentences (German → English):
- **Konflikte im Team lösen** → *resolve conflicts in the team* (our query)
- **Streit zwischen Kollegen schlichten** → *settle arguments between colleagues* (same meaning, **no shared words**)
- **Excel Formeln lernen** → *learn Excel formulas* (unrelated)

In [1]:
!pip -q install sentence-transformers

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

torch.set_grad_enabled(False)        # we only run the model, we never train it

def first(v, n=5):
    """Show only the first n of the 384 numbers, so the output stays readable."""
    return "[" + ", ".join(f"{x:+.3f}" for x in v[:n].tolist()) + ", ...]"

## Step 0 · Load the model

Two parts are downloaded:
- the **tokenizer**: the rules and the vocabulary for cutting text into pieces
- the **model**: the numbers (weights) that the training produced

`attn_implementation="eager"` only means "use the plain version of attention", so that we are allowed to look at the attention weights in step 4b.

In [2]:
NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(NAME)
model = AutoModel.from_pretrained(NAME, attn_implementation="eager")
model.eval()

cfg = model.config
n_params = sum(p.numel() for p in model.parameters())
print("Layers (transformer blocks):", cfg.num_hidden_layers)
print("Numbers per token vector:   ", cfg.hidden_size)
print("Attention heads per layer:  ", cfg.num_attention_heads)
print("Vocabulary size (pieces):   ", tokenizer.vocab_size)
print(f"Parameters (all weights):    {n_params/1e6:.0f} million")
emb_params = model.embeddings.word_embeddings.weight.numel()
print(f"  of which the lookup table: {emb_params/1e6:.0f} million  (vocabulary x {cfg.hidden_size})")

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Layers (transformer blocks): 12
Numbers per token vector:    384
Attention heads per layer:   12
Vocabulary size (pieces):    250002
Parameters (all weights):    118 million
  of which the lookup table: 96 million  (vocabulary x 384)


## Step 1 · The query

In [3]:
QUERY = "Konflikte im Team lösen"          # resolve conflicts in the team
print(QUERY)

Konflikte im Team lösen


## Step 2 · Tokenization: text → pieces → numbers

The model cannot read letters. The tokenizer cuts the text into **pieces from a fixed vocabulary** and gives each piece its **id number**.

How to read the output:
- `<s>` and `</s>` are special **start** and **end** markers that are always added.
- `▁` (a low line) means **"a new word starts here"**. A piece without `▁` is the rest of the previous word.
- A long or rare German word is cut into several pieces. That's how the model handles words it has never seen as a whole.

In [4]:
enc = tokenizer(QUERY, return_tensors="pt", truncation=True, max_length=128)   # 128 = the model's limit
ids = enc["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(ids)

print(f"{'pos':>3}  {'piece':<14} id")
for pos, (tok, i) in enumerate(zip(tokens, ids.tolist())):
    print(f"{pos:>3}  {tok:<14} {i}")
print(f"\n{len(QUERY.split())} words -> {len(tokens)} tokens")

# try your own:  print(tokenizer.tokenize("Gehaltsverhandlung"))   # salary negotiation

pos  piece          id
  0  <s>            0
  1  ▁Konflikt      126499
  2  e              13
  3  ▁im            566
  4  ▁Team          11399
  5  ▁lösen         182709
  6  </s>           2

4 words -> 7 tokens


In [14]:
print(tokenizer.tokenize("Gehaltsverhandlung"))   # salary negotiation

['▁Ge', 'halt', 's', 'ver', 'hand', 'lung']


## Step 3 · Embedding lookup: each id → its starting vector

The model has a big **table**: one row per vocabulary piece, 384 numbers per row.
The "lookup" is nothing more than **taking row number `id`** from that table.

At this point the vectors **know nothing about the sentence**. The piece "Team" gets exactly the same starting vector in every sentence in the world.

In [5]:
table = model.embeddings.word_embeddings.weight          # the lookup table
print("Table size:", tuple(table.shape), " (rows = pieces, columns = numbers per piece)\n")

raw = table[ids]                                          # pick one row per token id
for tok, vec in zip(tokens, raw):
    print(f"{tok:<14} {first(vec)}")

# proof that it really is just "take a row":
same = torch.equal(model.embeddings.word_embeddings(enc["input_ids"])[0], raw)
print("\nlookup layer == taking rows by hand:", same)

Table size: (250037, 384)  (rows = pieces, columns = numbers per piece)

<s>            [-0.008, -0.001, -0.005, -0.037, -0.010, ...]
▁Konflikt      [+0.020, +0.001, -0.013, -0.008, -0.012, ...]
e              [-0.012, -0.027, +0.029, +0.007, -0.001, ...]
▁im            [+0.018, -0.003, +0.009, -0.001, -0.032, ...]
▁Team          [-0.002, -0.020, -0.019, -0.017, +0.021, ...]
▁lösen         [+0.029, +0.036, +0.002, -0.020, +0.008, ...]
</s>           [-0.011, +0.011, +0.002, -0.003, -0.005, ...]

lookup layer == taking rows by hand: True


### Step 3b · Adding the position

Word order matters ("Hund beißt Mann" = *dog bites man* vs "Mann beißt Hund" = *man bites dog*).
So the model **adds a second vector that says the position** (1st, 2nd, 3rd…) and then rescales everything to a standard size (this is called *LayerNorm*).
The result is the input to layer 1. The model calls it `hidden_states[0]`.

In [6]:
out = model(**enc, output_hidden_states=True, output_attentions=True)
hs = out.hidden_states          # 13 snapshots: [0] = after step 3b, [1]..[12] = after each layer
print("Snapshots:", len(hs), "  each has shape", tuple(hs[0].shape), "(1 sentence, tokens, 384)\n")

for tok, before, after in zip(tokens, raw, hs[0][0]):
    print(f"{tok:<14} lookup only {first(before, 3)}   + position, rescaled {first(after, 3)}")

Snapshots: 13   each has shape (1, 7, 384) (1 sentence, tokens, 384)

<s>            lookup only [-0.008, -0.001, -0.005, ...]   + position, rescaled [-0.056, +0.021, +0.008, ...]
▁Konflikt      lookup only [+0.020, +0.001, -0.013, ...]   + position, rescaled [+0.654, +0.134, -0.829, ...]
e              lookup only [-0.012, -0.027, +0.029, ...]   + position, rescaled [-0.618, -1.006, +0.940, ...]
▁im            lookup only [+0.018, -0.003, +0.009, ...]   + position, rescaled [+0.614, -0.222, +0.246, ...]
▁Team          lookup only [-0.002, -0.020, -0.019, ...]   + position, rescaled [+0.130, -0.699, -0.709, ...]
▁lösen         lookup only [+0.029, +0.036, +0.002, ...]   + position, rescaled [+1.187, +1.564, +0.141, ...]
</s>           lookup only [-0.011, +0.011, +0.002, ...]   + position, rescaled [-0.318, +0.204, +0.259, ...]


## Step 4 · The 12 transformer layers

Each layer does two things to **every** token vector:
1. **Attention**: the token looks at all the other tokens in the sentence and mixes in information from the ones that matter to it.
2. **Feed-forward**: a small neural network reworks that vector on its own.

The output of layer 1 is the input of layer 2, and so on, 12 times.
Below we follow **one token** through all 12 layers.

- **numbers** = the first 5 of its 384 numbers. Watch them change.
- **similar to previous** = cosine similarity with the same token one layer earlier. 1.00 = unchanged, lower = this layer changed it more.
- **similar to start** = how far it has moved from its starting vector.

In [7]:
# pick the token to follow: the first piece that contains "team" (change it to follow another one)
WATCH = next(i for i, t in enumerate(tokens) if "team" in t.lower())
print(f"Following token #{WATCH}: {tokens[WATCH]!r}\n")

start = hs[0][0, WATCH]
print(f"{'layer':<9} {'numbers (first 5 of 384)':<52} {'similar to previous':>20} {'similar to start':>17}")
print(f"{'input':<9} {first(start):<52}")
for L in range(1, len(hs)):
    v, prev = hs[L][0, WATCH], hs[L - 1][0, WATCH]
    sim_prev = F.cosine_similarity(v, prev, dim=0).item()
    sim_start = F.cosine_similarity(v, start, dim=0).item()
    print(f"{'layer ' + str(L):<9} {first(v):<52} {sim_prev:>20.2f} {sim_start:>17.2f}")

Following token #4: '▁Team'

layer     numbers (first 5 of 384)                              similar to previous  similar to start
input     [+0.130, -0.699, -0.709, -0.492, +0.633, ...]       
layer 1   [+0.278, -0.657, -0.741, +0.085, +0.309, ...]                        0.94              0.94
layer 2   [+0.257, -0.399, -1.004, +0.413, +0.236, ...]                        0.96              0.88
layer 3   [+0.396, -0.568, -1.019, +0.367, +0.134, ...]                        0.96              0.81
layer 4   [-0.078, -0.784, -0.830, +0.495, -0.159, ...]                        0.94              0.72
layer 5   [-0.394, -0.766, -0.361, +0.434, -0.331, ...]                        0.94              0.65
layer 6   [-0.412, -0.588, +0.021, +0.459, -0.468, ...]                        0.95              0.56
layer 7   [-0.614, -1.015, -0.294, +0.134, -0.509, ...]                        0.94              0.49
layer 8   [-0.868, -1.139, -0.032, +0.099, -0.173, ...]                        0.94         

### Step 4b · Attention: which words does our token look at?

In every layer, each token gives **attention weights** to all tokens (they add up to 100 %).
This layer has several **heads** (parallel "viewpoints"); we show their average.
Compare an early layer with the last one: the pattern changes as the model builds up meaning.

In [8]:
def show_attention(layer):
    w = out.attentions[layer - 1][0].mean(0)[WATCH]      # average over heads, row of our token
    print(f"Layer {layer}: where does {tokens[WATCH]!r} look?")
    for tok, a in zip(tokens, w.tolist()):
        print(f"   {tok:<14} {a*100:5.1f} %  {'█' * round(a * 40)}")
    print()

show_attention(1)
show_attention(cfg.num_hidden_layers)

Layer 1: where does '▁Team' look?
   <s>             30.4 %  ████████████
   ▁Konflikt       20.1 %  ████████
   e                4.5 %  ██
   ▁im              5.1 %  ██
   ▁Team           11.6 %  █████
   ▁lösen          12.3 %  █████
   </s>            16.0 %  ██████

Layer 12: where does '▁Team' look?
   <s>             27.6 %  ███████████
   ▁Konflikt        6.0 %  ██
   e                3.4 %  █
   ▁im              3.9 %  ██
   ▁Team           22.1 %  █████████
   ▁lösen           9.3 %  ████
   </s>            27.6 %  ███████████



### Step 4c · Watch the meaning form, layer by layer

Now take **whole sentences**. For every layer we turn the sentence into one vector (steps 5 and 6, explained below) and compare:
- query vs **Streit zwischen Kollegen schlichten** (*settle arguments between colleagues*): same meaning, no shared words
- query vs **Excel Formeln lernen** (*learn Excel formulas*): unrelated

**What to look for:** in the early layers everything looks fairly similar, because the vectors still mostly reflect the individual pieces. The gap between "related" and "unrelated" should be clearest in the **last layer**, because that is the layer the model was trained to get right.

In [9]:
def sentence_vector_per_layer(text):
    e = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    o = model(**e, output_hidden_states=True)
    mask = e["attention_mask"][0].unsqueeze(-1).float()
    return [F.normalize((h[0] * mask).sum(0) / mask.sum(), dim=0) for h in o.hidden_states]

RELATED   = "Streit zwischen Kollegen schlichten"     # settle arguments between colleagues
UNRELATED = "Excel Formeln lernen"                    # learn Excel formulas
q, r, u = (sentence_vector_per_layer(t) for t in (QUERY, RELATED, UNRELATED))

print(f"{'layer':<8} {'vs related':>11} {'vs unrelated':>13} {'gap':>7}")
for L in range(len(q)):
    sr, su = (q[L] @ r[L]).item(), (q[L] @ u[L]).item()
    print(f"{'input' if L == 0 else L:<8} {sr:>11.2f} {su:>13.2f} {sr - su:>+7.2f}")

layer     vs related  vs unrelated     gap
input           0.46          0.37   +0.09
1               0.50          0.40   +0.09
2               0.59          0.50   +0.09
3               0.62          0.51   +0.11
4               0.64          0.48   +0.16
5               0.70          0.47   +0.23
6               0.74          0.48   +0.26
7               0.77          0.50   +0.26
8               0.79          0.52   +0.27
9               0.82          0.51   +0.31
10              0.81          0.39   +0.42
11              0.78          0.23   +0.55
12              0.70         -0.00   +0.70


## Step 5 · Mean pooling: many token vectors → one sentence vector

After layer 12 we have **one vector per token**. Search needs **one vector per sentence**.
The simplest way: the **average** of all token vectors (number by number).

The `mask` says which tokens are real. With one sentence they all are; when several sentences are processed together, the shorter ones get padded with empty tokens, and the mask keeps those out of the average.

In [10]:
last = hs[-1][0]                                   # after layer 12: (tokens, 384)
mask = enc["attention_mask"][0].unsqueeze(-1).float()

print("Token vectors after layer 12:")
for tok, vec in zip(tokens, last):
    print(f"   {tok:<14} {first(vec)}")

pooled = (last * mask).sum(0) / mask.sum()         # add them up, divide by how many
print(f"\nAverage (sentence)  {first(pooled)}   shape: {tuple(pooled.shape)}")

# by hand, for the first number only:
col = [round(x, 3) for x in last[:, 0].tolist()]
print(f"\nFirst number, by hand: ({' + '.join(map(str, col))}) / {len(col)} = {sum(col)/len(col):.3f}")

Token vectors after layer 12:
   <s>            [+0.087, +0.014, +0.017, -0.247, +0.047, ...]
   ▁Konflikt      [+0.020, +0.037, -0.189, +0.300, -0.188, ...]
   e              [-0.050, +0.208, -0.167, +0.144, -0.045, ...]
   ▁im            [-0.300, +0.155, -0.311, -0.082, +0.110, ...]
   ▁Team          [-0.597, -0.535, +0.013, -0.287, +0.073, ...]
   ▁lösen         [+0.254, +0.226, -0.025, +0.043, -0.203, ...]
   </s>           [+0.085, +0.015, +0.018, -0.244, +0.043, ...]

Average (sentence)  [-0.072, +0.017, -0.092, -0.053, -0.023, ...]   shape: (384,)

First number, by hand: (0.087 + 0.02 + -0.05 + -0.3 + -0.597 + 0.254 + 0.085) / 7 = -0.072


## Step 6 · Normalization: make the length 1

The **length** of a vector is √(x₁² + x₂² + … + x₃₈₄²). We divide every number by the length, so the new length is exactly 1.

Why: with length-1 vectors, the simple **dot product** (multiply number by number, then add) **is** the cosine similarity. That's why the app can score all coaches with one `@`.

In [11]:
length = pooled.norm().item()
sentence_vec = pooled / length
print(f"Length before: {length:.3f}")
print(f"Length after:  {sentence_vec.norm().item():.3f}")
print(f"\nFinal query vector (first 5 of 384): {first(sentence_vec)}")

Length before: 4.568
Length after:  1.000

Final query vector (first 5 of 384): [-0.016, +0.004, -0.020, -0.012, -0.005, ...]


## Step 7 · Check: is this exactly what the app gets?

The app calls `SentenceTransformer(...).encode(text, normalize_embeddings=True)`.
If our step-by-step vector matches it, we have rebuilt the whole one-liner by hand.

In [12]:
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer(NAME)
app_vec = torch.tensor(st_model.encode(QUERY, normalize_embeddings=True))

diff = (app_vec - sentence_vec).abs().max().item()
print(f"Biggest difference between the two vectors: {diff:.2e}")
print("Same vector as the app:", torch.allclose(app_vec, sentence_vec, atol=1e-4))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Biggest difference between the two vectors: 5.12e-08
Same vector as the app: True


## Step 8 · Compare: the actual search

One vector per coach sentence, then a dot product with the query, then sort. That's `vector_rank()` in `search.py`.
Note the English sentence: the multilingual training put it close to the German query.

In [13]:
COACH_SENTENCES = {
    "Streit zwischen Kollegen schlichten": "settle arguments between colleagues",
    "Spannungen im Team abbauen":          "reduce tension in the team",
    "resolving conflicts within teams":    "(English sentence)",
    "Präsentationstraining für Vertrieb":  "presentation training for sales",
    "Excel Formeln lernen":                "learn Excel formulas",
}
coach_vecs = torch.tensor(st_model.encode(list(COACH_SENTENCES), normalize_embeddings=True))
scores = coach_vecs @ app_vec                    # one dot product per coach sentence

print(f"Query: {QUERY}  (resolve conflicts in the team)\n")
for (de, en), s in sorted(zip(COACH_SENTENCES.items(), scores.tolist()), key=lambda x: -x[1]):
    print(f"{s:6.2f}  {de:<38} {en}")

Query: Konflikte im Team lösen  (resolve conflicts in the team)

  0.94  resolving conflicts within teams       (English sentence)
  0.82  Spannungen im Team abbauen             reduce tension in the team
  0.70  Streit zwischen Kollegen schlichten    settle arguments between colleagues
  0.21  Präsentationstraining für Vertrieb     presentation training for sales
 -0.00  Excel Formeln lernen                   learn Excel formulas


## Summary: each step and where it lives in the app

| Step | Here | In `search.py` |
|---|---|---|
| 2 Tokenization | `tokenizer(QUERY)` | inside `model.encode(...)` |
| 3 Embedding lookup | `table[ids]` | inside `model.encode(...)` |
| 4 12 layers | `model(**enc)` | inside `model.encode(...)` |
| 5 Mean pooling | `(last * mask).sum(0) / mask.sum()` | inside `model.encode(...)` |
| 6 Normalization | `pooled / length` | `normalize_embeddings=True` |
| 8 Compare | `coach_vecs @ app_vec` | `doc_vectors() @ embed([query])[0]` in `vector_rank()` |

**Try:** change `QUERY` in step 1 (for example `"Gehalt verhandeln"`, *negotiate salary*), then Runtime → Run all.